In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())
print(os.getcwd())

C:\Users\alvee\Documents\ai-grid-fault-classifier


In [2]:
%%writefile common.py
import pandas as pd

TARGET = "fault_types"
FEATURES = ["Ia","Ib","Ic","Va","Vb","Vc","Im","Vm","Pai","Pav","Fi","Fv"]
CLASSES = ["NF","LG","LL","LLG","LLL","LLLG"]
RANDOM_STATE = 42

def load_clean(path="data/processed/clean.csv"):
    return pd.read_csv(path)

def blocked_split(df, n_blocks=5):
    df = df.copy()
    n = df.groupby(TARGET)[TARGET].transform("size")
    df["block"] = df.groupby(TARGET).cumcount() * n_blocks // n
    train = df[df["block"] < n_blocks - 1]   # first 80% of every run
    test = df[df["block"] == n_blocks - 1]   # last 20% of every run
    return train, test

Writing common.py


In [3]:
from common import *

df = pd.read_csv("data/raw/PMU_Fault_Data.csv")

In [4]:
df.head()

,time_stamp,Ia,Ib,Ic,Va,Vb,Vc,Im,Vm,Pai,Pav,Fi,Fv,fault_types
0,0.000000e+00,1963.798615,-8897.985690,6934.187075,64.010540,-285.098287,221.087747,1.0,1.0,0.0,0.0,50.0,50.0,NF
1,3.160000e-30,1963.798615,-8897.985690,6934.187075,64.010540,-285.098287,221.087747,1.0,1.0,0.0,0.0,50.0,50.0,NF
2,3.125000e-04,2849.984363,-9136.332309,6286.347925,92.337771,-292.611899,200.274127,1.0,1.0,0.0,0.0,50.0,50.0,NF
3,6.250000e-04,3708.810435,-9286.742028,5577.931566,119.778538,-297.309131,177.530592,1.0,1.0,0.0,0.0,50.0,50.0,NF
4,9.375000e-04,4531.982598,-9347.752715,4815.770110,146.067826,-299.144311,153.076485,1.0,1.0,0.0,0.0,50.0,50.0,NF


In [5]:
print(df.shape, df.isna().sum().sum())      # (134406, 14) 0
print(df[TARGET].value_counts())            # 22,401 per class

df = df.drop_duplicates()                   # removes 17 rows
warmup = (df["Im"] == 1.0) & (df["Vm"] == 1.0)
df = df[~warmup].copy()                     # removes 370 rows
df = df.sort_values([TARGET, "time_stamp"], kind="stable").reset_index(drop=True)
df.to_csv("data/processed/clean.csv", index=False)
print(df.shape)                             # (134019, 14)

train, test = blocked_split(df)
print(len(train), len(test))                

(134406, 14) 0
fault_types
NF      22401
LG      22401
LL      22401
LLG     22401
LLL     22401
LLLG    22401
Name: count, dtype: int64
(134019, 14)
107218 26801


In [6]:
%%writefile data_dictionary.md
# Data Dictionary: PMU_Fault_Data.csv

Source: ________ (add the exact dataset name or link from your faculty or download page)
Raw size: 134,406 rows x 14 columns. Cleaned size: 134,019 rows x 14 columns.

| Column | Meaning (inferred from name and values) | Notes |
|---|---|---|
| time_stamp | Time in seconds | Restarts at 0 for every class. Not used as a feature. |
| Ia, Ib, Ic | Currents of phases A, B, C | Range about -47,000 to +45,000 |
| Va, Vb, Vc | Voltages of phases A, B, C | Range about -307 to +306 |
| Im | Current magnitude | Nearly constant inside each class |
| Vm | Voltage magnitude | Nearly constant inside each class |
| Pai | Phase angle of current (degrees) | To be confirmed with the dataset source |
| Pav | Phase angle of voltage (degrees) | To be confirmed with the dataset source |
| Fi | Frequency from current (Hz) | About 50 Hz in every class, little information |
| Fv | Frequency from voltage (Hz) | About 50 Hz in every class, little information |
| fault_types | Target label | NF, LG, LL, LLG, LLL, LLLG |

The column meanings are inferred from names and values. They must be confirmed with the faculty or the data source before the final report.

## Classes
| Class | Meaning | Rows (raw) |
|---|---|---|
| NF | No fault (normal) | 22,401 |
| LG | Line to ground | 22,401 |
| LL | Line to line | 22,401 |
| LLG | Two lines to ground | 22,401 |
| LLL | All three lines shorted | 22,401 |
| LLLG | Three lines to ground | 22,401 |

## Features used for modeling
Ia, Ib, Ic, Va, Vb, Vc, Im, Vm, Pai, Pav, Fi, Fv (time_stamp excluded).

Writing data_dictionary.md


In [7]:
%%writefile README.md
# AI Grid Fault Classifier

Classifying normal operation and electrical fault conditions (NF, LG, LL, LLG, LLL, LLLG) from PMU voltage, current and frequency data, using Python, Decision Tree, Random Forest and SVM.

## Team
| Member | Name | Roll no. | Role | Branch |
|---|---|---|---|---|
| 1 | Alveera | ____ | Data Lead and Repository Owner | m1-data |
| 2 | ____ | ____ | Visualization and Electrical-Domain Lead | m2-viz |
| 3 | ____ | ____ | Modeling Lead | m3-model |
| 4 | ____ | ____ | Evaluation, Robustness and Documentation Lead | m4-eval |

## Folder structure
- data/raw/PMU_Fault_Data.csv : original data, never edited
- data/processed/clean.csv : cleaned data (duplicates and warm-up rows removed)
- common.py : shared feature list, labels, random_state and split function
- notebooks/ : 01_data_cleaning, 02_visualization, 03_modeling, 04_evaluation
- figures/ : saved plots
- models/ : saved models
- report/ : report and slides
- data_dictionary.md : column meanings

## How to run
1. Clone the repo and create an environment: conda create -n gridfault python=3.11
2. Activate it and run: pip install -r requirements.txt
3. Start Jupyter from the repo root folder: jupyter notebook
4. Run the notebooks in order (01 to 04). Every notebook starts with: from common import *

## Team rules
- One person, one notebook, one branch. Only edit your own files.
- common.py is changed only by Member 1.
- Split data by time blocks (blocked_split), never by random rows.
- Never use time_stamp as a feature. random_state = 42 everywhere.
- Every change goes through a Pull Request reviewed by another member.
- Review chain: 1 reviews 2, 2 reviews 3, 3 reviews 4, 4 reviews 1.
- Run git pull before starting work each day.

Writing README.md


In [8]:
%%writefile cleaning_notes.md
# Data cleaning summary (Member 1)

The raw file has 134,406 rows and 14 columns, with no missing values and 22,401 rows in each of the 6 classes.

Three cleaning steps were applied:
1. 17 fully duplicated rows were removed.
2. 370 warm-up rows were removed. These are placeholder rows at the start of each class run, where Im = 1, Vm = 1 and the phase angle is 0. They are not real fault measurements and would look like a fake extra pattern.
3. The data was sorted by class and time_stamp so that each class run is in time order, which the time-blocked split needs.

The cleaned file has 134,019 rows. The time-blocked split holds out the last 20 percent of every class run: 107,218 training rows and 26,801 test rows. A random row split is avoided because neighbouring samples are 0.3 ms apart and almost identical, which would leak test information into training and inflate the score.

time_stamp restarts at 0 in every class, so it is not used as a feature.

Writing cleaning_notes.md


In [9]:
import os
print([f for f in os.listdir(".") if f.endswith(".md")])

['cleaning_notes.md', 'data_dictionary.md', 'README.md']
